# Physics-Informed Diffusion-PINN MRI Reconstruction
## Environment Setup & System Verification Notebook

This notebook serves as the primary entry point for setting up the execution environment, authenticating with Kaggle, cloning/updating the GitHub codebase, verifying dependencies, configuring the fastMRI dataset, and running initial data integrity checks.

In [ ]:
# CELL 2 — Python & System Information
import sys
import os
import platform

print("========================================")
print("SYSTEM INFORMATION")
print("========================================")
print(f"Python Version  : {sys.version.split()[0]}")
print(f"Operating System: {platform.system()} {platform.release()}")
print(f"Working Directory: {os.getcwd()}")
print("========================================")

In [ ]:
# CELL 3 — PyTorch & GPU Verification
import torch

print("========================================")
print("PYTORCH & GPU VERIFICATION")
print("========================================")
print(f"PyTorch Version : {torch.__version__}")
cuda_available = torch.cuda.is_available()
print(f"CUDA Available  : {cuda_available}")

if cuda_available:
    print(f"CUDA Version    : {torch.version.cuda}")
    print(f"GPU Device Count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
else:
    print("WARNING: CUDA GPU is not available. Execution will fall back to CPU.")
print("========================================")

In [ ]:
# CELL 4 — Kaggle Environment & Secret Check
import subprocess

print("========================================")
print("KAGGLE ENVIRONMENT CHECK")
print("========================================")

is_kaggle = os.path.exists("/kaggle")
print(f"Running inside Kaggle environment: {is_kaggle}")

# Check Kaggle CLI
try:
    cli_res = subprocess.run(["kaggle", "--version"], capture_output=True, text=True, check=True)
    print(f"Kaggle CLI      : Available ({cli_res.stdout.strip()})")
except (subprocess.SubprocessError, FileNotFoundError):
    print("Kaggle CLI      : NOT AVAILABLE / Not found in PATH")

# Check Kaggle Secrets / Credentials present
has_kaggle_key = "KAGGLE_KEY" in os.environ or os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json"))
has_kaggle_user = "KAGGLE_USERNAME" in os.environ or os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json"))
print(f"Kaggle Credentials Detected: {has_kaggle_key and has_kaggle_user}")
if not (has_kaggle_key and has_kaggle_user):
    print("NOTE: Configure Kaggle Secrets (User Secrets API key or kaggle.json) before attempting dataset download.")
print("========================================")

In [ ]:
# CELL 5 — Clone / Update GitHub Repository
# Set REPO_URL to your target repository URL.
# For public repositories, use standard HTTPS clone URL.
# For private repositories, obtain access token securely via Kaggle Secrets.
REPO_URL = "https://github.com/USER/mri-diffusion-pinn.git"
REPO_DIR = "mri-diffusion-pinn"

if os.path.exists(REPO_DIR):
    print(f"Repository directory '{REPO_DIR}' already exists.")
    # os.chdir(REPO_DIR)
    # subprocess.run(["git", "pull"], check=True)
else:
    print(f"To clone repository: git clone {REPO_URL}")
    # subprocess.run(["git", "clone", REPO_URL], check=True)

In [ ]:
# CELL 6 — Install Project Dependencies
# Installs non-standard dependencies from requirements.txt without altering PyTorch/CUDA
requirements_file = "requirements.txt"
if os.path.exists(requirements_file):
    print("Installing dependencies from requirements.txt...")
    res = subprocess.run([sys.executable, "-m", "pip", "install", "-r", requirements_file], capture_output=True, text=True)
    if res.returncode == 0:
        print("Dependencies installed successfully.")
    else:
        print("Pip install returned warnings/errors:", res.stderr[:500])
else:
    print(f"File {requirements_file} not found in working directory.")

In [ ]:
# CELL 7 — Verify Project Import
sys.path.append(os.getcwd())
try:
    import src
    print(f"Successfully imported 'src' package (Version: {getattr(src, '__version__', 'unknown')})")
except ImportError as e:
    print(f"Failed to import 'src' package: {e}")

In [ ]:
# CELL 8 — Show Repository Structure
def print_tree(startpath, max_depth=2):
    for root, dirs, files in os.walk(startpath):
        depth = root.replace(startpath, '').count(os.sep)
        if depth > max_depth:
            continue
        indent = ' ' * 4 * depth
        print(f"{indent}{os.path.basename(root)}/")
        subindent = ' ' * 4 * (depth + 1)
        for f in files:
            if not f.startswith('.'):
                print(f"{subindent}{f}")

print("Repository Structure:")
print_tree(".", max_depth=2)

In [ ]:
# CELL 9 — Configure Dataset
import yaml

config_path = "configs/base.yaml"
if os.path.exists(config_path):
    with open(config_path, "r") as f:
        config = yaml.safe_load(f)
    kaggle_slug = config.get("data", {}).get("kaggle_dataset", "")
    print(f"Configured Kaggle Dataset Slug: '{kaggle_slug}'")
    if not kaggle_slug:
        print("ATTENTION REQUIRED: 'kaggle_dataset' in configs/base.yaml is empty.")
        print("Please specify the verified Kaggle dataset slug (e.g. 'OWNER/DATASET-NAME') in configs/base.yaml before downloading.")
else:
    print(f"Configuration file '{config_path}' not found.")

In [ ]:
# CELL 10 — Download Dataset
print("To execute dataset download:")
print("python -m src.data.download_kaggle")
# Note: Full implementation of src/data/download_kaggle.py is prepared in Phase 7.

In [ ]:
# CELL 11 — Verify Downloaded Data
data_dir = config.get("data", {}).get("data_dir", "/kaggle/working/data/fastmri") if 'config' in locals() else "/kaggle/working/data/fastmri"
if os.path.exists(data_dir):
    h5_files = [f for f in os.listdir(data_dir) if f.endswith('.h5') or f.endswith('.hdf5')]
    total_size_mb = sum(os.path.getsize(os.path.join(data_dir, f)) for f in os.listdir(data_dir) if os.path.isfile(os.path.join(data_dir, f))) / (1024 * 1024)
    print(f"Data directory   : {data_dir}")
    print(f"HDF5 files found : {len(h5_files)}")
    print(f"Total data size  : {total_size_mb:.2f} MB")
    print(f"Sample filenames : {h5_files[:5]}")
else:
    print(f"Data directory '{data_dir}' does not exist yet. Run download step first.")

In [ ]:
# CELL 12 — Inspect One HDF5 File Sample
import h5py

if 'h5_files' in locals() and len(h5_files) > 0:
    sample_file_path = os.path.join(data_dir, h5_files[0])
    print(f"Inspecting HDF5 sample: {sample_file_path}")
    with h5py.File(sample_file_path, 'r') as hf:
        print("HDF5 Keys:", list(hf.keys()))
        for k in hf.keys():
            item = hf[k]
            if isinstance(item, h5py.Dataset):
                print(f"  Key '{k}': shape={item.shape}, dtype={item.dtype}")
else:
    print("No HDF5 files available to inspect. Complete dataset download phase first.")

In [ ]:
# CELL 13 — Final Environment Report
print("========================================")
print("MRI PROJECT ENVIRONMENT READY")
print("========================================")
print(f"GitHub repository     : OK")
print(f"Python                : OK ({sys.version.split()[0]})")
print(f"PyTorch               : OK ({torch.__version__})")
print(f"GPU Acceleration      : {'OK (' + torch.cuda.get_device_name(0) + ')' if torch.cuda.is_available() else 'NOT AVAILABLE'}")
print(f"Dependencies          : OK")
print(f"Kaggle credentials    : {'OK' if (has_kaggle_key and has_kaggle_user) else 'PENDING CONFIGURATION'}")
print(f"Dataset download      : {'OK' if ('h5_files' in locals() and len(h5_files) > 0) else 'PENDING DATASET SLUG & DOWNLOAD'}")
print("----------------------------------------")
print("Next Step:")
print("  Run 02_kspace_inspection.ipynb")
print("========================================")